# Análisis de causas de falla operativa

Este notebook analiza 120 registros de falla agrupados por causa. El objetivo es identificar las causas con mayor incidencia y establecer prioridades de atención mediante un diagrama de Pareto y una distribución porcentual.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Permite ejecutar el notebook desde la raíz del proyecto o desde notebooks/.
project_root = Path.cwd()
if not (project_root / "data" / "causas_falla_frecuencias.csv").exists():
    project_root = project_root.parent

figures_dir = project_root / "figuras"
figures_dir.mkdir(exist_ok=True)
print(f"Figuras PNG: {figures_dir.resolve()}")


## Análisis de causas de falla operativa

Este bloque analiza 120 registros agrupados por causa de falla. La fuente `causas_falla_frecuencias.csv` contiene la consolidación de frecuencias obtenida a partir de esos registros.


In [ ]:
failure_path = project_root / "data" / "causas_falla_frecuencias.csv"
assert failure_path.exists(), f"No se encontró el archivo: {failure_path}"

failures = pd.read_csv(failure_path)
failures["Frecuencia"] = failures["Frecuencia"].astype(int)
failures = failures.sort_values("Frecuencia", ascending=False).reset_index(drop=True)
total_failures = failures["Frecuencia"].sum()

assert total_failures == 120, "La tabla de causas debe contener 120 registros"
failures["Porcentaje"] = failures["Frecuencia"] / total_failures * 100
failures["Frecuencia acumulada"] = failures["Frecuencia"].cumsum()
failures["% acumulado"] = failures["Porcentaje"].cumsum()

print(f"Registros analizados: {total_failures}")
display(failures)


### Diagrama de Pareto

El diagrama de Pareto ordena las causas de mayor a menor frecuencia y muestra el porcentaje acumulado. Es útil para priorizar las causas que concentran la mayor parte de las incidencias.


In [ ]:
pareto_80_index = int(np.argmax(failures["% acumulado"].to_numpy() >= 80))

fig, ax_frequency = plt.subplots(figsize=(13, 6))

positions = np.arange(len(failures))
bars = ax_frequency.bar(positions, failures["Frecuencia"], color="#247BA0")
ax_frequency.set(
    title="Diagrama de Pareto de causas de falla",
    xlabel="Causa",
    ylabel="Frecuencia",
)
ax_frequency.set_xticks(positions)
ax_frequency.set_xticklabels(failures["Causa"], rotation=35, ha="right", fontsize=10)
ax_frequency.tick_params(axis="y", labelsize=10)
ax_frequency.set_title("Diagrama de Pareto de causas de falla", fontsize=14, fontweight="semibold", pad=12)
ax_frequency.set_xlabel("Causa", fontsize=11)
ax_frequency.set_ylabel("Frecuencia", fontsize=11)
ax_frequency.set_axisbelow(True)
ax_frequency.grid(axis="y", alpha=0.28, linewidth=0.8)
ax_frequency.grid(axis="x", visible=False)
ax_frequency.margins(y=0.12)

for bar, value in zip(bars, failures["Frecuencia"]):
    ax_frequency.text(bar.get_x() + bar.get_width() / 2, value + 0.6, str(value), ha="center", va="bottom")

ax_percentage = ax_frequency.twinx()
ax_percentage.plot(positions, failures["% acumulado"], color="#D1495B", marker="o", markersize=6, linewidth=2.4, label="% acumulado")
ax_percentage.axhline(80, color="#555555", linestyle="--", linewidth=1.2, label="80%")
ax_percentage.scatter(positions[pareto_80_index], failures.loc[pareto_80_index, "% acumulado"], color="#D1495B", edgecolor="white", linewidth=1, s=65, zorder=4)
ax_percentage.annotate("80%", xy=(positions[pareto_80_index], failures.loc[pareto_80_index, "% acumulado"]), xytext=(8, 10), textcoords="offset points", color="#555555", fontsize=10, fontweight="semibold")
ax_percentage.set_ylabel("Porcentaje acumulado", fontsize=11)
ax_percentage.set_ylim(0, 100)
ax_percentage.set_yticks(np.arange(0, 101, 20))
ax_percentage.tick_params(axis="y", labelsize=10)
ax_percentage.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax_percentage.legend(loc="lower right")

fig.tight_layout()
fig.subplots_adjust(bottom=0.28)
fig.savefig(figures_dir / "05_pareto_causas_falla.png", dpi=300, bbox_inches="tight")
plt.show()

leading_cause = failures.iloc[0]
pareto_80_index = int(np.argmax(failures["% acumulado"].to_numpy() >= 80))
pareto_80_count = pareto_80_index + 1
pareto_80_pct = failures.loc[pareto_80_index, "% acumulado"]
display(Markdown(f"""
**Interpretaciones**

1. **{leading_cause['Causa']}** es la causa más frecuente, con {leading_cause['Frecuencia']} registros ({leading_cause['Porcentaje']:.1f}% del total).
2. Las primeras {pareto_80_count} causas acumulan {pareto_80_pct:.1f}% de las incidencias; atenderlas primero concentra el esfuerzo donde puede generar mayor impacto.
3. La línea acumulada permite distinguir prioridades: las causas al inicio del gráfico tienen mayor peso operativo que las ubicadas al final.
"""))


### Distribución porcentual de causas

El gráfico de pastel muestra la participación de cada causa dentro del total de incidencias y complementa la priorización del diagrama de Pareto.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.Set3(np.linspace(0, 1, len(failures)))
wedges, _, percentages = ax.pie(
    failures["Frecuencia"],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)
ax.set_title("Distribución porcentual de causas de falla", fontsize=14, fontweight="semibold", pad=12)
ax.legend(
    wedges,
    failures["Causa"],
    title="Causas",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize=10,
    frameon=False,
)
ax.set_aspect("equal")
fig.tight_layout(rect=[0, 0, 0.76, 1])
fig.savefig(figures_dir / "06_pastel_causas_falla.png", dpi=300, bbox_inches="tight")
plt.show()

smallest_cause = failures.iloc[-1]
top_five_pct = failures.loc[:4, "Porcentaje"].sum()
display(Markdown(f"""
**Interpretaciones**

1. El sector más grande corresponde a **{leading_cause['Causa']}**, que representa {leading_cause['Porcentaje']:.1f}% de los registros.
2. Las cinco causas con mayor participación reúnen {top_five_pct:.1f}% del total, por lo que la distribución está concentrada en un grupo reducido de problemas.
3. **{smallest_cause['Causa']}** representa solo {smallest_cause['Porcentaje']:.1f}% de las incidencias; su baja participación no elimina la necesidad de monitorearla, pero su prioridad relativa es menor.
"""))


## Síntesis del análisis

La distribución de frecuencias permite distinguir las causas prioritarias de las de menor incidencia. El Pareto concentra la atención en las causas con mayor impacto, mientras que el gráfico de pastel facilita comunicar la participación porcentual de cada categoría.
